# 00 - Save the best YOLOv8n model and export it for the Jetson  (run on Colab)

**What this notebook does**
1. Re-creates your NEU-DET train/val split exactly (same seed), so the numbers match your experiment notebook.
2. Loads the model you chose: **`combined_scale_translate_mosaic`** (best recall at the same mAP50 as the original checkpoint).
3. Measures precision, recall, F1, AP50 and AP50-95 per class, and finds the confidence threshold to deploy.
4. Saves a **deployment bundle** to Google Drive: weights, `model_meta.json`, metric tables, label-size statistics, demo images.
5. Exports **ONNX** (ONNX Runtime is the vision runtime on the Jetson) and checks that it matches the PyTorch model.

**Outputs** (in `MyDrive/neu_det_yolov8n/deploy/`): `yolov8n_neu_combined.pt`, `yolov8n_neu_combined.onnx`, `model_meta.json`,
`metrics_val_classwise.csv`, `confidence_sweep.csv`, `label_area_stats.csv`, `demo_images/`.

Runtime: GPU is not required for this notebook, but it is faster with one.

In [ ]:

%pip -q install -U "ultralytics>=8.3.0" onnx onnxruntime onnxslim pyyaml pandas matplotlib

import os, json, random, shutil, zipfile, datetime
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, yaml
import xml.etree.ElementTree as ET
import ultralytics
from ultralytics import YOLO
from google.colab import drive
drive.mount("/content/drive")

WORK = Path("/content/drive/MyDrive/neu_det_yolov8n")
RUNS = WORK / "runs"
PROJECT = RUNS / "supervisor_experiments"
DEPLOY = WORK / "deploy"
DEPLOY.mkdir(parents=True, exist_ok=True)
DATA_YAML = Path("/content/neu_yolo/data.yaml")
CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
IMG_SIZE, BATCH, SEED = 416, 8, 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"

def extract_archive(zip_path, extract_to="/content/neu_yolo"):
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)

# Same dataset preparation as your experiment notebook (same shuffle seed => identical val split).
if not DATA_YAML.exists():
    local_zip = Path("/content/archive.zip")
    drive_zips = list(Path("/content/drive/MyDrive").rglob("archive.zip"))
    if local_zip.exists():
        extract_archive(local_zip)
    elif drive_zips:
        extract_archive(drive_zips[0])
    base_path = Path("/content/neu_yolo")
    ann_dir = (list(base_path.rglob("ANNOTATIONS")) + list(base_path.rglob("Annotations")))[0]
    img_dir = (list(base_path.rglob("IMAGES")) + list(base_path.rglob("Images")))[0]
    for subset in ["train", "val"]:
        os.makedirs(base_path / "images" / subset, exist_ok=True)
        os.makedirs(base_path / "labels" / subset, exist_ok=True)
    img_files = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.JPEG")))
    random.seed(42)
    random.shuffle(img_files)
    split_idx = int(len(img_files) * 0.85)
    splits = {"train": img_files[:split_idx], "val": img_files[split_idx:]}
    for subset, files in splits.items():
        for img_path in files:
            shutil.copy(img_path, base_path / "images" / subset / img_path.name)
            xml_path = ann_dir / f"{img_path.stem}.xml"
            if xml_path.exists():
                root = ET.parse(xml_path).getroot()
                size = root.find("size")
                w, h = int(size.find("width").text), int(size.find("height").text)
                with open(base_path / "labels" / subset / f"{img_path.stem}.txt", "w") as lf:
                    for obj in root.iter("object"):
                        cname = obj.find("name").text.lower().replace(" ", "_")
                        if cname in CLASSES:
                            bb = obj.find("bndbox")
                            x1, x2 = float(bb.find("xmin").text), float(bb.find("xmax").text)
                            y1, y2 = float(bb.find("ymin").text), float(bb.find("ymax").text)
                            lf.write(f"{CLASSES.index(cname)} {(x1+x2)/(2*w):.6f} {(y1+y2)/(2*h):.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}\n")
    with open(DATA_YAML, "w") as f:
        yaml.safe_dump({"path": "/content/neu_yolo", "train": "images/train", "val": "images/val",
                        "names": {i: n for i, n in enumerate(CLASSES)}}, f)
assert DATA_YAML.exists(), "data.yaml missing - check that archive.zip is on your Drive"
print("Dataset ready:", DATA_YAML, "| GPU:", torch.cuda.is_available())

In [ ]:

CHOSEN_NAME = "combined_scale_translate_mosaic"
COMBINED = PROJECT / CHOSEN_NAME / "weights" / "best.pt"
assert COMBINED.exists(), f"Not found: {COMBINED}. Check the folder name in {PROJECT}"
model = YOLO(str(COMBINED))
NAMES = [model.names[i] for i in sorted(model.names)]
assert NAMES == CLASSES, f"class order differs: {NAMES}"
print("Loaded", COMBINED, "\nClasses:", NAMES)

## 3) Metrics and the deployment confidence threshold

Your earlier threshold-sweep cell printed precision/recall as 0.0 because of how the confusion matrix was indexed. Here the precision, recall and F1
are read from Ultralytics' per-class curves (`p_curve`, `r_curve`, `f1_curve`), which are computed from the validation predictions at conf = 0.001.

* **`CONF_F1`** is the confidence that maximises mean F1 across classes (balanced choice).
* **`CONF_RECALL`** is the highest confidence that still keeps macro recall >= `RECALL_TARGET` (use it if missing a defect is worse than a false alarm).
* Whichever you choose is saved as `deploy_conf`. This choice is made on the **validation** split.

In [ ]:
RECALL_TARGET = 0.75

val = model.val(data=str(DATA_YAML), split="val", imgsz=IMG_SIZE, batch=BATCH, device=DEVICE, conf=0.001,
                plots=True, project=str(DEPLOY / "val_plots"), name=CHOSEN_NAME, exist_ok=True, verbose=False)
b = val.box
cls_idx = np.asarray(b.ap_class_index, dtype=int)
px = np.asarray(b.px).reshape(-1)
Pc, Rc, F1c = np.asarray(b.p_curve), np.asarray(b.r_curve), np.asarray(b.f1_curve)   # (n_classes, 1000)
mean_f1 = F1c.mean(0)

best_i = int(np.nanargmax(mean_f1))
CONF_F1 = float(px[best_i])
ok = np.where(Rc.mean(0) >= RECALL_TARGET)[0]
CONF_RECALL = float(px[ok.max()]) if len(ok) else None

def at(conf):
    i = int(np.abs(px - conf).argmin())
    return Pc[:, i], Rc[:, i], F1c[:, i]

sweep = []
for thr in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]:
    p, r, f = at(thr)
    sweep.append({"confidence": thr, "precision_macro": p.mean(), "recall_macro": r.mean(), "f1_macro": f.mean()})
sweep_df = pd.DataFrame(sweep).round(4)
display(sweep_df)
print(f"CONF_F1 (max mean F1) = {CONF_F1:.3f}   |   CONF_RECALL (recall >= {RECALL_TARGET}) = {CONF_RECALL}")

plt.figure(figsize=(7, 3.6))
plt.plot(px, Pc.mean(0), label="precision"); plt.plot(px, Rc.mean(0), label="recall"); plt.plot(px, mean_f1, label="F1")
plt.axvline(CONF_F1, ls="--", c="grey"); plt.xlabel("confidence threshold"); plt.ylabel("macro metric")
plt.title("Validation: precision, recall, F1 vs confidence"); plt.grid(alpha=.3); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Choose the threshold the Jetson will use (default: balanced F1). Change to CONF_RECALL if you prefer recall.
DEPLOY_CONF = round(CONF_F1, 3)
p, r, f = at(DEPLOY_CONF)
rows = []
for k, ci in enumerate(cls_idx):
    rows.append({"class": CLASSES[ci], "precision": p[k], "recall": r[k], "F1": f[k],
                 "AP50": float(b.ap50[k]), "AP50_95": float(b.ap[k])})
classwise = pd.DataFrame(rows).round(4)
display(classwise)
overall = {"mAP50": float(b.map50), "mAP50_95": float(b.map), "precision_macro": float(p.mean()),
           "recall_macro": float(r.mean()), "f1_macro": float(f.mean()), "conf": DEPLOY_CONF}
print(json.dumps({k: round(v, 4) for k, v in overall.items()}, indent=2))

## 4) Test split check (important)

Your experiment notebook builds only **train (85%) and val (15%)**: `data.yaml` has no `test` entry. So there is **no untouched test set**, and the
validation images were used to pick the model and threshold. Report these numbers as **validation** results (not test results) in your report, or
re-split 70/15/15 and retrain if your supervisor requires a held-out test. The cell below evaluates a test split only if one exists.

In [ ]:
cfg_y = yaml.safe_load(open(DATA_YAML))
test_metrics = None
if "test" in cfg_y:
    t = model.val(data=str(DATA_YAML), split="test", imgsz=IMG_SIZE, batch=BATCH, device=DEVICE, conf=DEPLOY_CONF,
                  plots=False, verbose=False)
    test_metrics = {"mAP50": float(t.box.map50), "mAP50_95": float(t.box.map)}
    print("TEST (run once, do not tune on it):", test_metrics)
else:
    print("No test split in data.yaml -> reporting validation metrics only.")

## 5) Save the deployment bundle to Drive

In [ ]:
shutil.copy(COMBINED, DEPLOY / "yolov8n_neu_combined.pt")
classwise.to_csv(DEPLOY / "metrics_val_classwise.csv", index=False)
sweep_df.to_csv(DEPLOY / "confidence_sweep.csv", index=False)

# Label-size statistics (used in notebook 03 to sanity-check the severity size bands)
rows = []
for f in Path("/content/neu_yolo/labels/train").glob("*.txt"):
    for line in f.read_text().splitlines():
        if line.strip():
            c, x, y, w, h = line.split()
            rows.append((CLASSES[int(c)], float(w) * float(h)))
area_stats = (pd.DataFrame(rows, columns=["class", "area_frac"]).groupby("class")["area_frac"]
              .describe(percentiles=[.1, .5, .9]).round(4))
area_stats.to_csv(DEPLOY / "label_area_stats.csv")
display(area_stats)

# A few val images per class for the offline demo
img_dir, lab_dir = Path("/content/neu_yolo/images/val"), Path("/content/neu_yolo/labels/val")
picked = {}
for lf in sorted(lab_dir.glob("*.txt")):
    for cid in {int(l.split()[0]) for l in lf.read_text().splitlines() if l.strip()}:
        if len(picked.setdefault(cid, [])) < 5:
            picked[cid].append(lf.stem)
(DEPLOY / "demo_images").mkdir(exist_ok=True)
for stem in sorted({s for v in picked.values() for s in v}):
    for src in img_dir.glob(stem + ".*"):
        shutil.copy(src, DEPLOY / "demo_images" / src.name)
print("demo images:", len(list((DEPLOY / "demo_images").glob("*"))))

meta = {
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "selected_model": CHOSEN_NAME, "source_checkpoint": str(COMBINED),
    "classes": CLASSES, "imgsz": IMG_SIZE, "deploy_conf": DEPLOY_CONF,
    "conf_f1_optimal": round(CONF_F1, 3), "conf_recall_priority": CONF_RECALL, "recall_target": RECALL_TARGET,
    "nms_iou": 0.5, "ultralytics_version": ultralytics.__version__,
    "split": "train 85% / val 15%, seed 42 (no test split)",
    "val_metrics": {k: round(v, 4) for k, v in overall.items()},
    "test_metrics": test_metrics,
    "classwise_val": classwise.to_dict("records"),
}
(DEPLOY / "model_meta.json").write_text(json.dumps(meta, indent=2))
print("saved to", DEPLOY)

## 6) Export to ONNX and verify

`simplify=True` uses onnxslim; `opset=12` is conservative so older ONNX Runtime builds on JetPack can load it. The static 416x416 input is intentional (faster, simpler on the Jetson).
The verification re-validates the **ONNX** file with Ultralytics: mAP should match the PyTorch numbers to about +/- 0.005.

In [ ]:
onnx_path = Path(YOLO(str(COMBINED)).export(format="onnx", imgsz=IMG_SIZE, opset=12, simplify=True,
                                             dynamic=False, half=False, device="cpu"))
print("exported:", onnx_path, f"{onnx_path.stat().st_size/1e6:.1f} MB")

onnx_val = YOLO(str(onnx_path), task="detect").val(data=str(DATA_YAML), split="val", imgsz=IMG_SIZE, batch=1,
                                                    conf=0.001, device="cpu", plots=False, verbose=False)
print(f"PyTorch  mAP50={b.map50:.4f}  mAP50-95={b.map:.4f}")
print(f"ONNX     mAP50={onnx_val.box.map50:.4f}  mAP50-95={onnx_val.box.map:.4f}")
assert abs(onnx_val.box.map50 - b.map50) < 0.01, "ONNX and PyTorch disagree - do not deploy until this is understood"

shutil.copy(onnx_path, DEPLOY / "yolov8n_neu_combined.onnx")
import onnxruntime as ort
s = ort.InferenceSession(str(DEPLOY / "yolov8n_neu_combined.onnx"), providers=["CPUExecutionProvider"])
print("input", s.get_inputs()[0].name, s.get_inputs()[0].shape, "| output", s.get_outputs()[0].shape)

## 7) Download the bundle (copy it to the Jetson repo's `deploy/` folder)

In [ ]:
zip_path = shutil.make_archive(str(WORK / "deploy_bundle"), "zip", DEPLOY)
print("bundle:", zip_path, f"{Path(zip_path).stat().st_size/1e6:.1f} MB")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download manually from Drive:", zip_path)